# Testing QuestTreeClassifier

This notebook tests the `questtree` module on real-world datasets:

1. **Iris Dataset** - Classic multiclass classification (continuous features)
2. **UCI Car Evaluation Dataset** - Multiclass with categorical/ordinal features

---

## Setup

In [1]:
import numpy as np
import pandas as pd
from questtree import (
    QuestTreeClassifier,
    print_tree,
    get_tree_summary,
    get_feature_importances,
    classification_report,
    accuracy_score,
    confusion_matrix
)

# For reproducibility
np.random.seed(42)

print("questtree module loaded successfully!")

questtree module loaded successfully!


---
## Test 1: Iris Dataset

The Iris dataset is a classic multiclass classification problem with:
- 150 samples
- 4 continuous features (sepal/petal length/width)
- 3 classes (setosa, versicolor, virginica)

This tests QUEST's:
- Multiclass handling via Super-Class Clustering (Algorithm 4)
- ANOVA-based variable selection for continuous features
- QDA split point selection

In [2]:
# Load Iris dataset (manually, no sklearn dependency)
# Using the classic Iris data

iris_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"

try:
    iris_df = pd.read_csv(iris_url, header=None, names=[
        'sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class'
    ])
    print("Loaded Iris dataset from UCI repository")
except:
    # Fallback: create synthetic iris-like data
    print("Creating synthetic Iris-like dataset...")
    np.random.seed(42)
    n_per_class = 50
    
    # Class 0: Setosa (small petals)
    setosa = np.column_stack([
        np.random.normal(5.0, 0.35, n_per_class),  # sepal_length
        np.random.normal(3.4, 0.38, n_per_class),  # sepal_width
        np.random.normal(1.5, 0.17, n_per_class),  # petal_length
        np.random.normal(0.2, 0.10, n_per_class),  # petal_width
    ])
    
    # Class 1: Versicolor (medium)
    versicolor = np.column_stack([
        np.random.normal(5.9, 0.52, n_per_class),
        np.random.normal(2.8, 0.31, n_per_class),
        np.random.normal(4.3, 0.47, n_per_class),
        np.random.normal(1.3, 0.20, n_per_class),
    ])
    
    # Class 2: Virginica (large)
    virginica = np.column_stack([
        np.random.normal(6.6, 0.64, n_per_class),
        np.random.normal(3.0, 0.32, n_per_class),
        np.random.normal(5.5, 0.55, n_per_class),
        np.random.normal(2.0, 0.27, n_per_class),
    ])
    
    X_iris = np.vstack([setosa, versicolor, virginica])
    y_iris_labels = ['Iris-setosa']*50 + ['Iris-versicolor']*50 + ['Iris-virginica']*50
    
    iris_df = pd.DataFrame(X_iris, columns=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'])
    iris_df['class'] = y_iris_labels

print(f"\nDataset shape: {iris_df.shape}")
print(f"\nClass distribution:")
print(iris_df['class'].value_counts())
iris_df.head()

Loaded Iris dataset from UCI repository

Dataset shape: (150, 5)

Class distribution:
class
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64


,sepal_length,sepal_width,petal_length,petal_width,class
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [3]:
# Prepare data
X_iris = iris_df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].values
y_iris = iris_df['class'].values

# Train-test split (80-20)
n_samples = len(y_iris)
indices = np.random.permutation(n_samples)
train_size = int(0.8 * n_samples)

train_idx = indices[:train_size]
test_idx = indices[train_size:]

X_train_iris = X_iris[train_idx]
X_test_iris = X_iris[test_idx]
y_train_iris = y_iris[train_idx]
y_test_iris = y_iris[test_idx]

print(f"Training samples: {len(y_train_iris)}")
print(f"Test samples: {len(y_test_iris)}")

Training samples: 120
Test samples: 30


In [4]:
# Train QUEST classifier on Iris
clf_iris = QuestTreeClassifier(random_state=42)
clf_iris.fit(X_train_iris, y_train_iris)

print("=" * 60)
print("QUEST Classifier - Iris Dataset")
print("=" * 60)
print(f"\nDetected feature types: {clf_iris.feature_types_}")
print(f"Number of classes: {clf_iris.n_classes_}")
print(f"Classes: {list(clf_iris.classes_)}")
print(f"\nTree structure:")
print(f"  - Number of leaves: {clf_iris.get_n_leaves()}")
print(f"  - Maximum depth: {clf_iris.get_depth()}")

QUEST Classifier - Iris Dataset

Detected feature types: ['continuous', 'continuous', 'continuous', 'continuous']
Number of classes: 3
Classes: ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']

Tree structure:
  - Number of leaves: 5
  - Maximum depth: 4


In [5]:
# Evaluate on Iris
y_pred_iris = clf_iris.predict(X_test_iris)

train_acc = clf_iris.score(X_train_iris, y_train_iris)
test_acc = clf_iris.score(X_test_iris, y_test_iris)

print(f"\nAccuracy:")
print(f"  - Training: {train_acc:.4f}")
print(f"  - Test: {test_acc:.4f}")

print(f"\n{classification_report(y_test_iris, y_pred_iris)}")


Accuracy:
  - Training: 0.9333
  - Test: 0.9000

Classification Report
          Class    Precision       Recall     F1-Score    Support
------------------------------------------------------------
    Iris-setosa       0.7778       1.0000       0.8750          7
Iris-versicolor       1.0000       0.8182       0.9000         11
 Iris-virginica       0.9167       0.9167       0.9167         12
------------------------------------------------------------
      macro avg       0.8981       0.9116       0.8972         30
   weighted avg       0.9148       0.9000       0.9008         30

Accuracy: 0.9000


In [6]:
# Print tree structure
feature_names_iris = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
print("\nTree Structure:")
print_tree(clf_iris, feature_names=feature_names_iris)


Tree Structure:
QUEST Decision Tree Structure:
  Classes: ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
  Leaves: 5
  Depth: 4
[Node] sepal_length (feature 0) (continuous)
       Super-classes: ['Iris-setosa'] vs ['Iris-versicolor', 'Iris-virginica']
       Threshold: 5.3016
       n_samples: 120
  |-- Left (<=):
  |   [LEAF] Predict: Iris-setosa | n=39 | dist=(Iris-setosa:35, Iris-versicolor:4)
  |-- Right (>):
      [Node] petal_length (feature 2) (continuous)
             Super-classes: ['Iris-versicolor', 'Iris-virginica'] vs ['Iris-setosa']
             Threshold: 2.0119
             n_samples: 81
        |-- Left (<=):
        |   [LEAF] Predict: Iris-setosa | n=8 | dist=(Iris-setosa:8)
        |-- Right (>):
            [Node] petal_width (feature 3) (continuous)
                   Super-classes: ['Iris-versicolor'] vs ['Iris-virginica']
                   Threshold: 1.6303
                   n_samples: 73
              |-- Left (<=):
              |   [Node] petal_lengt

In [7]:
# Feature importances
importances_iris = get_feature_importances(clf_iris, feature_names_iris)
print("\nFeature Importances (by split frequency):")
print("-" * 40)
for name, imp in sorted(importances_iris.items(), key=lambda x: -x[1]):
    bar = "*" * int(imp * 40)
    print(f"{name:15s}: {imp:.4f} {bar}")


Feature Importances (by split frequency):
----------------------------------------
petal_length   : 0.5000 ********************
sepal_length   : 0.2500 **********
petal_width    : 0.2500 **********
sepal_width    : 0.0000 


In [8]:
# Test with pruning
print("\nEffect of Cost-Complexity Pruning on Iris:")
print("-" * 60)
print(f"{'Alpha':>8} {'Leaves':>8} {'Train Acc':>12} {'Test Acc':>12}")
print("-" * 60)

for alpha in [0.0, 0.01, 0.02, 0.05, 0.1]:
    clf_temp = QuestTreeClassifier(ccp_alpha=alpha, random_state=42)
    clf_temp.fit(X_train_iris, y_train_iris)
    
    train_acc = clf_temp.score(X_train_iris, y_train_iris)
    test_acc = clf_temp.score(X_test_iris, y_test_iris)
    
    print(f"{alpha:>8.2f} {clf_temp.get_n_leaves():>8d} {train_acc:>12.4f} {test_acc:>12.4f}")


Effect of Cost-Complexity Pruning on Iris:
------------------------------------------------------------
   Alpha   Leaves    Train Acc     Test Acc
------------------------------------------------------------
    0.00        5       0.9333       0.9000
    0.01        4       0.9250       0.9000
    0.02        4       0.9250       0.9000
    0.05        4       0.9250       0.9000
    0.10        4       0.9250       0.9000


---
## Test 2: UCI Car Evaluation Dataset

The Car Evaluation dataset is a multiclass classification problem with:
- 1728 samples
- 6 categorical/ordinal features
- 4 classes (unacc, acc, good, vgood)

Features:
- buying: vhigh, high, med, low
- maint: vhigh, high, med, low
- doors: 2, 3, 4, 5more
- persons: 2, 4, more
- lug_boot: small, med, big
- safety: low, med, high

This tests QUEST's:
- CRIMCOORDS transformation for categorical variables (Algorithm 2)
- Chi-square test for categorical variable selection
- Multiclass handling with 4 classes

In [9]:
# Load UCI Car Evaluation dataset
car_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data"

try:
    car_df = pd.read_csv(car_url, header=None, names=[
        'buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class'
    ])
    print("Loaded Car Evaluation dataset from UCI repository")
except Exception as e:
    print(f"Could not load from URL: {e}")
    print("Creating synthetic Car Evaluation-like dataset...")
    
    np.random.seed(42)
    n_samples = 1728
    
    car_df = pd.DataFrame({
        'buying': np.random.choice(['vhigh', 'high', 'med', 'low'], n_samples),
        'maint': np.random.choice(['vhigh', 'high', 'med', 'low'], n_samples),
        'doors': np.random.choice(['2', '3', '4', '5more'], n_samples),
        'persons': np.random.choice(['2', '4', 'more'], n_samples),
        'lug_boot': np.random.choice(['small', 'med', 'big'], n_samples),
        'safety': np.random.choice(['low', 'med', 'high'], n_samples),
    })
    
    # Create class based on rules
    def assign_class(row):
        if row['safety'] == 'low' or row['persons'] == '2':
            return 'unacc'
        elif row['buying'] in ['vhigh', 'high'] and row['maint'] in ['vhigh', 'high']:
            return 'unacc'
        elif row['buying'] == 'low' and row['maint'] == 'low' and row['safety'] == 'high':
            return 'vgood'
        elif row['safety'] == 'high' and row['lug_boot'] == 'big':
            return 'good'
        else:
            return 'acc'
    
    car_df['class'] = car_df.apply(assign_class, axis=1)

print(f"\nDataset shape: {car_df.shape}")
print(f"\nClass distribution:")
print(car_df['class'].value_counts())
print(f"\nFeature values:")
for col in car_df.columns[:-1]:
    print(f"  {col}: {list(car_df[col].unique())}")
car_df.head(10)

Loaded Car Evaluation dataset from UCI repository

Dataset shape: (1728, 7)

Class distribution:
class
unacc    1210
acc       384
good       69
vgood      65
Name: count, dtype: int64

Feature values:
  buying: ['vhigh', 'high', 'med', 'low']
  maint: ['vhigh', 'high', 'med', 'low']
  doors: ['2', '3', '4', '5more']
  persons: ['2', '4', 'more']
  lug_boot: ['small', 'med', 'big']
  safety: ['low', 'med', 'high']


,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc
5,vhigh,vhigh,2,2,med,high,unacc
6,vhigh,vhigh,2,2,big,low,unacc
7,vhigh,vhigh,2,2,big,med,unacc
8,vhigh,vhigh,2,2,big,high,unacc
9,vhigh,vhigh,2,4,small,low,unacc


In [10]:
# Prepare data (keep as categorical strings)
feature_cols = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety']
X_car = car_df[feature_cols].values
y_car = car_df['class'].values

# Train-test split (80-20)
n_samples = len(y_car)
indices = np.random.permutation(n_samples)
train_size = int(0.8 * n_samples)

train_idx = indices[:train_size]
test_idx = indices[train_size:]

X_train_car = X_car[train_idx]
X_test_car = X_car[test_idx]
y_train_car = y_car[train_idx]
y_test_car = y_car[test_idx]

print(f"Training samples: {len(y_train_car)}")
print(f"Test samples: {len(y_test_car)}")
print(f"\nTraining class distribution:")
unique, counts = np.unique(y_train_car, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {u}: {c}")

Training samples: 1382
Test samples: 346

Training class distribution:
  acc: 302
  good: 57
  unacc: 977
  vgood: 46


In [11]:
# Train QUEST classifier on Car Evaluation
# Note: All features are categorical - QUEST will auto-detect this
clf_car = QuestTreeClassifier(random_state=42)
clf_car.fit(X_train_car, y_train_car)

print("=" * 60)
print("QUEST Classifier - Car Evaluation Dataset")
print("=" * 60)
print(f"\nDetected feature types: {clf_car.feature_types_}")
print(f"Number of classes: {clf_car.n_classes_}")
print(f"Classes: {list(clf_car.classes_)}")
print(f"\nTree structure:")
print(f"  - Number of leaves: {clf_car.get_n_leaves()}")
print(f"  - Maximum depth: {clf_car.get_depth()}")

QUEST Classifier - Car Evaluation Dataset

Detected feature types: ['categorical', 'categorical', 'categorical', 'categorical', 'categorical', 'categorical']
Number of classes: 4
Classes: ['acc', 'good', 'unacc', 'vgood']

Tree structure:
  - Number of leaves: 51
  - Maximum depth: 9


In [12]:
# Evaluate on Car dataset
y_pred_car = clf_car.predict(X_test_car)

train_acc = clf_car.score(X_train_car, y_train_car)
test_acc = clf_car.score(X_test_car, y_test_car)

print(f"\nAccuracy:")
print(f"  - Training: {train_acc:.4f}")
print(f"  - Test: {test_acc:.4f}")

print(f"\n{classification_report(y_test_car, y_pred_car)}")


Accuracy:
  - Training: 0.9689
  - Test: 0.9335

Classification Report
          Class    Precision       Recall     F1-Score    Support
------------------------------------------------------------
            acc       0.8675       0.8780       0.8727         82
           good       0.5500       0.9167       0.6875         12
          unacc       0.9867       0.9528       0.9694        233
          vgood       1.0000       0.9474       0.9730         19
------------------------------------------------------------
      macro avg       0.8510       0.9237       0.8757        346
   weighted avg       0.9440       0.9335       0.9369        346

Accuracy: 0.9335


In [13]:
# Print tree structure (first few levels)
print("\nTree Structure:")
print_tree(clf_car, feature_names=feature_cols)


Tree Structure:
QUEST Decision Tree Structure:
  Classes: ['acc', 'good', 'unacc', 'vgood']
  Leaves: 51
  Depth: 9
[Node] persons (feature 3) (categorical)
       Super-classes: ['unacc'] vs ['acc', 'good', 'vgood']
       CRIMCOORDS threshold: 0.0002
       Left categories: ['2']
       Right categories: ['more', '4']
       n_samples: 1382
  |-- Left (<=):
  |   [LEAF] Predict: unacc | n=471 | dist=(unacc:471)
  |-- Right (>):
      [Node] safety (feature 5) (categorical)
             Super-classes: ['unacc'] vs ['acc', 'good', 'vgood']
             CRIMCOORDS threshold: 0.0002
             Left categories: ['low']
             Right categories: ['med', 'high']
             n_samples: 911
        |-- Left (<=):
        |   [LEAF] Predict: unacc | n=307 | dist=(unacc:307)
        |-- Right (>):
            [Node] buying (feature 0) (categorical)
                   Super-classes: ['acc'] vs ['good', 'unacc', 'vgood']
                   CRIMCOORDS threshold: -0.0000
                  

In [14]:
# Feature importances for Car dataset
importances_car = get_feature_importances(clf_car, feature_cols)
print("\nFeature Importances (by split frequency):")
print("-" * 40)
for name, imp in sorted(importances_car.items(), key=lambda x: -x[1]):
    bar = "*" * int(imp * 40)
    print(f"{name:12s}: {imp:.4f} {bar}")


Feature Importances (by split frequency):
----------------------------------------
lug_boot    : 0.3000 ************
safety      : 0.2600 **********
maint       : 0.1800 *******
buying      : 0.1200 ****
persons     : 0.0800 ***
doors       : 0.0600 **


In [15]:
# Test with pruning on Car dataset
print("\nEffect of Cost-Complexity Pruning on Car Evaluation:")
print("-" * 60)
print(f"{'Alpha':>8} {'Leaves':>8} {'Train Acc':>12} {'Test Acc':>12}")
print("-" * 60)

for alpha in [0.0, 0.005, 0.01, 0.02, 0.05]:
    clf_temp = QuestTreeClassifier(ccp_alpha=alpha, random_state=42)
    clf_temp.fit(X_train_car, y_train_car)
    
    train_acc = clf_temp.score(X_train_car, y_train_car)
    test_acc = clf_temp.score(X_test_car, y_test_car)
    
    print(f"{alpha:>8.3f} {clf_temp.get_n_leaves():>8d} {train_acc:>12.4f} {test_acc:>12.4f}")


Effect of Cost-Complexity Pruning on Car Evaluation:
------------------------------------------------------------
   Alpha   Leaves    Train Acc     Test Acc
------------------------------------------------------------
   0.000       51       0.9689       0.9335
   0.005       15       0.9052       0.8786
   0.010        6       0.8394       0.8092
   0.020        3       0.7815       0.7630
   0.050        1       0.7069       0.6734


---
## Test 3: Sklearn-like API Demonstration

In [16]:
# Demonstrate sklearn-like API
print("=" * 60)
print("Scikit-learn Compatible API Demonstration")
print("=" * 60)

# Create classifier
clf = QuestTreeClassifier(max_depth=5, ccp_alpha=0.01)

# get_params()
print("\n1. get_params():")
params = clf.get_params()
for k, v in params.items():
    print(f"   {k}: {v}")

# set_params()
print("\n2. set_params(max_depth=10, ccp_alpha=0.02):")
clf.set_params(max_depth=10, ccp_alpha=0.02)
print(f"   New max_depth: {clf.max_depth}")
print(f"   New ccp_alpha: {clf.ccp_alpha}")

# fit()
print("\n3. fit(X, y):")
clf.fit(X_train_iris, y_train_iris)
print(f"   Fitted! n_features_in_: {clf.n_features_in_}")

# predict()
print("\n4. predict(X):")
predictions = clf.predict(X_test_iris[:5])
print(f"   Predictions: {list(predictions)}")

# predict_proba()
print("\n5. predict_proba(X):")
proba = clf.predict_proba(X_test_iris[:3])
print(f"   Classes: {list(clf.classes_)}")
print(f"   Probabilities:")
for i, p in enumerate(proba):
    print(f"   Sample {i}: {[f'{x:.3f}' for x in p]}")

# score()
print("\n6. score(X, y):")
acc = clf.score(X_test_iris, y_test_iris)
print(f"   Accuracy: {acc:.4f}")

Scikit-learn Compatible API Demonstration

1. get_params():
   alpha: 0.05
   max_depth: 5
   min_samples_split: 10
   min_samples_leaf: 5
   ccp_alpha: 0.01
   feature_types: None
   random_state: None

2. set_params(max_depth=10, ccp_alpha=0.02):
   New max_depth: 10
   New ccp_alpha: 0.02

3. fit(X, y):
   Fitted! n_features_in_: 4

4. predict(X):
   Predictions: [np.str_('Iris-versicolor'), np.str_('Iris-setosa'), np.str_('Iris-versicolor'), np.str_('Iris-versicolor'), np.str_('Iris-setosa')]

5. predict_proba(X):
   Classes: ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
   Probabilities:
   Sample 0: ['0.000', '0.917', '0.083']
   Sample 1: ['0.897', '0.103', '0.000']
   Sample 2: ['0.000', '0.917', '0.083']

6. score(X, y):
   Accuracy: 0.9000


---
## Test 4: Mixed Features (Continuous + Categorical)

In [17]:
# Create a mixed dataset
print("=" * 60)
print("Mixed Features Test (Continuous + Categorical)")
print("=" * 60)

np.random.seed(42)
n = 300

# Continuous features
feat_cont1 = np.concatenate([
    np.random.normal(2, 1, 150),
    np.random.normal(5, 1, 150)
])
feat_cont2 = np.random.randn(n)

# Categorical feature (correlated with class)
feat_cat = np.concatenate([
    np.random.choice(['A', 'B'], 150, p=[0.8, 0.2]),
    np.random.choice(['A', 'B'], 150, p=[0.2, 0.8])
])

# Ordinal feature
feat_ord = np.concatenate([
    np.random.choice(['low', 'medium', 'high'], 150, p=[0.6, 0.3, 0.1]),
    np.random.choice(['low', 'medium', 'high'], 150, p=[0.1, 0.3, 0.6])
])

X_mixed = np.column_stack([feat_cont1, feat_cont2, feat_cat, feat_ord])
y_mixed = np.array([0] * 150 + [1] * 150)

# Shuffle
idx = np.random.permutation(n)
X_mixed = X_mixed[idx]
y_mixed = y_mixed[idx]

# Split
X_train_mixed = X_mixed[:240]
X_test_mixed = X_mixed[240:]
y_train_mixed = y_mixed[:240]
y_test_mixed = y_mixed[240:]

# Train
clf_mixed = QuestTreeClassifier(random_state=42)
clf_mixed.fit(X_train_mixed, y_train_mixed)

print(f"\nDetected feature types: {clf_mixed.feature_types_}")
print(f"Number of leaves: {clf_mixed.get_n_leaves()}")
print(f"Train accuracy: {clf_mixed.score(X_train_mixed, y_train_mixed):.4f}")
print(f"Test accuracy: {clf_mixed.score(X_test_mixed, y_test_mixed):.4f}")

# Feature importances
mixed_names = ['continuous_1', 'continuous_2', 'categorical', 'ordinal']
importances = get_feature_importances(clf_mixed, mixed_names)
print("\nFeature Importances:")
for name, imp in sorted(importances.items(), key=lambda x: -x[1]):
    print(f"  {name}: {imp:.4f}")

Mixed Features Test (Continuous + Categorical)

Detected feature types: ['continuous', 'continuous', 'binary', 'categorical']
Number of leaves: 12
Train accuracy: 0.9542
Test accuracy: 0.9500

Feature Importances:
  continuous_1: 0.5455
  continuous_2: 0.2727
  categorical: 0.0909
  ordinal: 0.0909


---
## Summary

The `questtree` module successfully:

1. **Handles multiclass classification** via Super-Class Clustering (Algorithm 4)
2. **Automatically detects feature types** (continuous vs categorical)
3. **Applies CRIMCOORDS** for categorical variable transformation (Algorithm 2)
4. **Uses statistical tests** for unbiased variable selection (Algorithm 3)
5. **Finds optimal splits** using QDA (Algorithm 1)
6. **Supports cost-complexity pruning** for regularization
7. **Provides a scikit-learn compatible API** without sklearn dependency

In [18]:
print("\n" + "=" * 60)
print("All tests completed successfully!")
print("=" * 60)


All tests completed successfully!
